In [45]:
import numpy as np

np.random.seed(42)

In [50]:
X = np.array([
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1],
])

y = np.array([0, 1, 1, 0])

In [58]:
def sigmoid(w):
    return 1 / (1 + np.exp(-w))

def relu(w):
    return np.maximum(w, 0)

class NDK_DENSE:

    def __init__(self, n_inputs, n_neurons, activation_fn="sigmoid"):
        self.weights = np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
        self.activation_fn = activation_fn
    
    def feed(self, X):
        # Save Z for the backward pass
        self.Z = np.dot(X, self.weights) + self.biases 
        self.X = X 

        if self.activation_fn == "sigmoid":
            self.A = sigmoid(self.Z) 
        
        elif self.activation_fn == "relu":
            self.A = relu(self.Z)
        
        return self.A
    
    def binary_loss(self, y_pred, y):
        epsilon = 1e-9  
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    
    def backward(self, dA, learning_rate, is_output=False):
        # 1. Calculate Activation Derivative (dZ)
        if is_output and self.activation_fn == "sigmoid":
            dZ = dA   
        else:
            if self.activation_fn == "sigmoid":
                dZ = dA * (self.A * (1 - self.A))
            elif self.activation_fn == "relu":
                dZ = dA * (self.Z > 0)
        
        m = self.X.shape[0]
        
        # 2. Calculate Weight and Bias Gradients (dW, db)
        dW = (1/m) * np.dot(self.X.T, dZ)
        db = (1/m) * np.sum(dZ, axis=0, keepdims=True)
        
        # 3. Calculate Error to pass down to previous layers
        dA_prev = np.dot(dZ, self.weights.T)
        
        # 4. Update Weights
        self.weights -= learning_rate * dW
        self.biases -= learning_rate * db
        
        # 5. Return previous layer gradients
        return dA_prev
 


In [59]:
hidden = NDK_DENSE(2, 2, activation_fn="relu")
output = NDK_DENSE(2, 1, activation_fn="sigmoid")

# Forward
A1 = hidden.feed(X)
A2 = output.feed(A1)

# Loss
loss = output.binary_loss(A2, y)

# Backward
dz2 = A2 - y
dA1 = output.backward(dz2, learning_rate=0.5, is_output=True)
hidden.backward(dA1, learning_rate=0.5)

array([[ 0.        ,  0.        ],
       [-0.07118375,  0.45266514],
       [ 0.16508059, -0.01658339],
       [ 0.        ,  0.        ]])